# LLM KServe Benchmark Notebook

Use this notebook as an orchestration + analysis layer while keeping scripts as source of truth.

## 1) Configure paths and run settings

In [ ]:
from pathlib import Path
import json, subprocess, os, textwrap

REPO_ROOT = Path.cwd()
MODEL = 'qwen25-0.5b-instruct'
POLICY = 'hpa-cpu-baseline'
SCENARIO = 'conversation-sharegpt'
PROM_URL = 'http://localhost:9090'
MAX_IN_FLIGHT = '4'
DRAIN_TIMEOUT_SECONDS = '300'
BENCH_TIMEOUT_SECONDS = '30'

SHAREGPT_INPUT = '/path/to/sharegpt.json'  # <- change this
SHAREGPT_OUTPUT = 'configs/data/sharegpt_prompts_5k.jsonl'

env = os.environ.copy()
env.update({
    'PROM_URL': PROM_URL,
    'MAX_IN_FLIGHT': MAX_IN_FLIGHT,
    'DRAIN_TIMEOUT_SECONDS': DRAIN_TIMEOUT_SECONDS,
    'BENCH_TIMEOUT_SECONDS': BENCH_TIMEOUT_SECONDS,
})
print('Configured. Repo:', REPO_ROOT)


## 2) Prepare ShareGPT prompts JSONL

In [ ]:
cmd = [
    'python', 'scripts/benchmark/prepare_sharegpt_dataset.py',
    '--input', SHAREGPT_INPUT,
    '--output', SHAREGPT_OUTPUT,
    '--max-samples', '5000',
    '--min-prompt-tokens', '16',
    '--max-prompt-tokens', '256',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True, env=env)


## 3) Run one policy evaluation

In [ ]:
cmd = [
    'bash', 'scripts/benchmark/run_policy_eval.sh',
    '--model', MODEL,
    '--policy', POLICY,
    '--scenario', SCENARIO,
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True, env=env)


## 4) Run multi-policy study (optional)

In [ ]:
env_study = env.copy()
env_study.update({
    'SCENARIO_LIST': SCENARIO,
    'POLICY_LIST': 'hpa-cpu-baseline keda-waiting-requests keda-token-aware keda-token-cache-composite',
    'REPEATS': '3',
})
cmd = ['bash', 'scripts/benchmark/run_policy_study.sh', '--model', MODEL]
print('Running:', ' '.join(cmd))
# subprocess.run(cmd, check=True, env=env_study)
print('Uncomment subprocess.run(...) when ready to execute the full study.')


## 5) Load latest summary quickly

In [ ]:
from pathlib import Path
import json

base = Path('results/policy_eval/qwen25_05b_instruct')
if not base.exists():
    print('No policy_eval results yet.')
else:
    summaries = sorted(base.glob('**/summary.json'), key=lambda p: p.stat().st_mtime)
    if not summaries:
        print('No summary.json files found.')
    else:
        latest = summaries[-1]
        print('Latest summary:', latest)
        print(json.dumps(json.loads(latest.read_text()), indent=2))
